# 07 - Reconciliation Planner

Combines heuristic dependency analysis with generative AI to produce actionable migration plans when naming/structural changes are required.

**Flow:**
1. Pull unresolved naming violations from `scan_results`
2. For each, query `system.access.table_lineage` to map full dependency graph
3. Feed context to LLM → generates structured migration plan (priority, steps, risk, effort)
4. Bank plan to `migration_plans` control table
5. Optionally push tickets to Jira/external backlog via API

**Philosophy:** The system never auto-renames. It *proposes* and *tracks*. Humans approve, the system orchestrates.

In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import require_widget, validate_identifier, validate_model_name, validate_uuid, safe_table_ref
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("control_schema", "uc_hygiene")
dbutils.widgets.text("target_catalogs", "")
dbutils.widgets.text("jira_base_url", "")  # e.g. https://yourorg.atlassian.net
dbutils.widgets.text("jira_project_key", "")  # e.g. DATA
dbutils.widgets.text("max_plans_per_run", "20")
dbutils.widgets.text("enable_ai_plans", "true")   # set to "false" to skip AI plan generation
dbutils.widgets.text("model_name", "databricks-gemini-3-5-flash")
dbutils.widgets.text("feature_check_mode", "standard")  # "standard" or "elevated"
dbutils.widgets.text("enable_feature_checks", "true")   # "false" to skip entirely


catalog          = require_widget(dbutils, "catalog")
control_schema   = require_widget(dbutils, "control_schema")
target_catalogs  = [
    c.strip() for c in require_widget(dbutils, "target_catalogs").split(",") if c.strip()
]
jira_base_url    = dbutils.widgets.get("jira_base_url").strip()
jira_project_key = dbutils.widgets.get("jira_project_key").strip()
max_plans        = int(dbutils.widgets.get("max_plans_per_run") or "20")
enable_ai_plans  = dbutils.widgets.get("enable_ai_plans").strip().lower() == "true"
model_name             = dbutils.widgets.get("model_name") or "databricks-gemini-3-5-flash"
feature_check_mode     = dbutils.widgets.get("feature_check_mode").strip() or "standard"
enable_feature_checks  = dbutils.widgets.get("enable_feature_checks").strip().lower() == "true"
# ── SQL injection hardening: validate all interpolated identifiers ──────────
validate_identifier(catalog, "catalog")
validate_identifier(control_schema, "control_schema")
for _tc in target_catalogs:
    validate_identifier(_tc, "target_catalog")
if enable_ai_plans:
    validate_model_name(model_name)
control_fqn = safe_table_ref(catalog, control_schema)

# Jira API token comes exclusively from secret scope -- never pass via widget
try:
    jira_api_token = dbutils.secrets.get("uc-steward", "jira-api-token")
except Exception:
    jira_api_token = ""  # Jira disabled if secret scope not configured

print(f"Control schema:     {control_fqn}")
print(f"Target catalogs:    {target_catalogs}")
print(f"Jira:               {'enabled' if jira_base_url and jira_api_token else 'disabled'}")
print(f"AI plan generation: {'enabled' if enable_ai_plans else 'DISABLED (heuristic fallback only)'}")
print(f"Model: {model_name if enable_ai_plans else 'N/A'} | Max plans: {max_plans}")
import time as _t; _task_start = _t.time()


In [0]:
# ── Platform Feature Enablement Checks ───────────────────────────────────────
# Verifies that UC platform features (Predictive Optimization, Data Classification,
# Lakehouse Monitoring) are enabled for governed assets.
#
# Privilege-aware: if the SP lacks access to elevated system tables, those checks
# are SKIPPED (not FAILED) and the hygiene score adjusts automatically.

import time as _t
_task_start = _t.time()

feature_results = []  # Default empty — populated if checks are enabled

if enable_feature_checks:
    from lib.feature_checks import (
        CapabilityProbe,
        run_feature_checks,
        results_to_spark,
    )
    from lib.policy import load_policy
    from databricks.sdk import WorkspaceClient

    _policy = load_policy(spark, catalog, control_schema)

    # Probe system table access once (zero-cost queries, cached for session)
    probe = CapabilityProbe(spark)
    print(f"Probing system table access (mode={feature_check_mode})...")
    probe.probe_all()
    print(probe.summary())

    # Run all enabled feature checks
    try:
        _sdk = WorkspaceClient()
    except Exception:
        _sdk = None
        print("  ⚠  SDK unavailable — Lakehouse Monitoring SDK fallback disabled")

    feature_results = run_feature_checks(
        spark=spark,
        sdk=_sdk,
        probe=probe,
        policy=_policy,
        catalogs=target_catalogs,
        control_schema=control_schema,
        mode=feature_check_mode,
    )
else:
    print("  Feature checks DISABLED (enable_feature_checks=false)")

# Persist results to control table
if feature_results:
    from datetime import date as _date
    from pyspark.sql import functions as F
    _results_df = results_to_spark(spark, feature_results)
    _results_df = _results_df.withColumn("scan_date", F.lit(_date.today()))
    _results_df.createOrReplaceTempView("_tmp_feature_checks")

    # Overwrite today's results (idempotent re-runs)
    spark.sql(f"""
    MERGE INTO {safe_table_ref(catalog, control_schema, "feature_check_results")} tgt
    USING _tmp_feature_checks src
      ON tgt.check_name = src.check_name
      AND tgt.scope_name = src.scope_name
      AND tgt.scan_date = src.scan_date
    WHEN MATCHED THEN UPDATE SET
      tier       = src.tier,
      status     = src.status,
      scope_type = src.scope_type,
      message    = src.message,
      checked_at = src.checked_at
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"  Persisted {len(feature_results)} feature check results to {catalog}.{control_schema}.feature_check_results")
else:
    print("  No feature checks executed (all disabled or no catalogs specified)")

print(f"  Feature checks completed in {_t.time() - _task_start:.1f}s")

In [0]:
from datetime import date
import uuid, json, re

scan_date = date.today()

# Aggregate ALL unresolved violations per table — one plan per unique object.
# This is the core "ladder up": column violations roll into their table,
# multiple scan_types (naming + staleness + tags) become a single holistic plan.
objects_needing_plans = spark.sql(f"""
WITH existing_plans AS (
  SELECT catalog_name, schema_name, table_name
  FROM {safe_table_ref(catalog, control_schema, "migration_plans")}
  WHERE plan_status NOT IN ('completed', 'cancelled')
),
table_violations AS (
  SELECT
    catalog_name,
    schema_name,
    table_name,
    COLLECT_SET(finding_type)   AS violation_types,
    COLLECT_SET(finding_detail) AS violation_details,
    MAX(CASE finding_severity
          WHEN 'critical' THEN 3
          WHEN 'warning'  THEN 2
          ELSE 1 END)           AS max_severity_rank,
    MIN(scan_date)              AS first_seen
  FROM {safe_table_ref(catalog, control_schema, "scan_results")}
  WHERE resolved_at IS NULL
  GROUP BY catalog_name, schema_name, table_name
)
SELECT tv.*,
  CASE max_severity_rank WHEN 3 THEN 'critical' WHEN 2 THEN 'warning' ELSE 'info' END AS max_severity
FROM table_violations tv
LEFT JOIN existing_plans ep
  ON tv.catalog_name = ep.catalog_name
  AND tv.schema_name = ep.schema_name
  AND tv.table_name  = ep.table_name
WHERE ep.table_name IS NULL
ORDER BY max_severity_rank DESC, first_seen ASC
LIMIT {max_plans}
""").collect()

print(f"Objects needing plans: {len(objects_needing_plans)}")
for o in objects_needing_plans[:5]:
    print(f"  {o.catalog_name}.{o.schema_name}.{o.table_name}  issues={list(o.violation_types)}")


In [0]:
# Lineage via UC REST API — indexed point-lookups, not table scans
# /api/2.1/lineage-tracking/table-lineage returns pre-computed lineage instantly

import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from databricks.sdk import WorkspaceClient

_wc = WorkspaceClient()
_host = _wc.config.host.rstrip("/")
_headers = _wc.config.authenticate()

def _fetch_lineage_api(fqn):
    """Single UC lineage API call for one table. Returns (fqn, upstream[], downstream[])."""
    try:
        resp = requests.get(
            f"{_host}/api/2.1/lineage-tracking/table-lineage",
            headers=_headers,
            params={"table_name": fqn, "include_entity_lineage": "true"},
            timeout=10
        )
        if resp.status_code != 200:
            return fqn, [], []
        data = resp.json()
        upstream   = data.get("upstream_tables",   [])
        downstream = data.get("downstream_tables", [])
        return fqn, upstream, downstream
    except Exception as e:
        print(f"  ⚠ Lineage API failed for {fqn}: {e}")
        return fqn, [], []


def build_lineage_map(objects):
    """
    Parallel UC lineage API calls — one per table, all in-flight simultaneously.
    No table scans. Each call is an indexed point-lookup: typically <200ms.
    """
    fqns = [f"{o.catalog_name}.{o.schema_name}.{o.table_name}" for o in objects]
    if not fqns:
        return {}

    result = {}
    with ThreadPoolExecutor(max_workers=min(len(fqns), 10)) as pool:
        futures = {pool.submit(_fetch_lineage_api, fqn): fqn for fqn in fqns}
        for future in as_completed(futures):
            fqn, upstream, downstream = future.result()

            consumer_types = {}
            for e in downstream:
                t = (e.get("entity_type") or "UNKNOWN").upper()
                consumer_types[t] = consumer_types.get(t, 0) + 1

            all_owners = list(set(
                [e.get("created_by") for e in downstream if e.get("created_by")] +
                [e.get("created_by") for e in upstream   if e.get("created_by")]
            ))

            result[fqn] = {
                "downstream":            [{"entity_type": (e.get("entity_type") or "UNKNOWN").upper(),
                                           "entity_id":   str(e.get("entity_id", "")),
                                           "dependent_table": e.get("table_name", ""),
                                           "owner": e.get("created_by")} for e in downstream[:20]],
                "upstream":              [{"entity_type": (e.get("entity_type") or "UNKNOWN").upper(),
                                           "entity_id":   str(e.get("entity_id", "")),
                                           "source_table": e.get("table_name", ""),
                                           "owner": e.get("created_by")} for e in upstream[:20]],
                "consumer_types":        consumer_types,
                "downstream_count":      len(downstream),
                "upstream_count":        len(upstream),
                "total_impacted_owners": all_owners,
            }

    print(f"  Lineage loaded for {len(result)}/{len(fqns)} tables via UC API")
    return result

print("build_lineage_map (UC REST API) ready.")


In [0]:
# Impact classifier + deterministic step builder
# Risk, priority, and steps are pure functions of lineage — no LLM needed here.
# External tool registry inflates risk for tables with JDBC/BI consumers invisible to UC lineage.

_STALENESS_VIOLATIONS = {"stale_table", "no_recent_access", "staleness_violation", "stale_warning", "stale_critical"}
_TAG_VIOLATIONS       = {"tag_violation", "missing_tags", "tag_compliance_violation", "missing_required_tag"}

# ── Load external tool registry (tables with known JDBC/BI consumers) ────────
_external_tool_map = {}  # fqn -> list of tool names
try:
    _ext_rows = spark.sql(f"""
    SELECT table_fqn, tool_name
    FROM {safe_table_ref(catalog, control_schema, "external_tool_registry")}
    WHERE is_active = true
    """).collect()
    for r in _ext_rows:
        _external_tool_map.setdefault(r.table_fqn, []).append(r.tool_name)
    if _external_tool_map:
        print(f"  External tool registry: {len(_external_tool_map)} tables with registered external consumers")
except Exception as e:
    print(f"  ⚠ External tool registry not available (non-fatal): {e}")


def classify_impact(dep_graph, fqn=None):
    types = dep_graph["consumer_types"]
    ds    = dep_graph["downstream_count"]
    has_pipeline  = bool(types.get("PIPELINE", 0)  + types.get("DLT_PIPELINE", 0))
    has_job       = bool(types.get("JOB", 0))
    has_dashboard = bool(types.get("DASHBOARD", 0))
    has_external  = bool(_external_tool_map.get(fqn)) if fqn else False
    ext_tools     = _external_tool_map.get(fqn, [])

    # External tools are invisible to lineage — treat as high risk
    if has_pipeline or has_dashboard or has_external:
        risk = "critical"
    elif has_job or ds > 5:
        risk = "high"
    elif ds > 0:
        risk = "medium"
    else:
        risk = "low"
    priority = min(100, int(
        20 + min(ds * 5, 40)
        + (25 if has_pipeline  else 0)
        + (20 if has_dashboard else 0)
        + (10 if has_job       else 0)
        + (30 if has_external  else 0)  # external tools = invisible blast radius
    ))
    return {
        "risk_level": risk, "priority_score": float(priority),
        "consumer_types": types,
        "has_pipeline": has_pipeline, "has_job": has_job,
        "has_dashboard": has_dashboard, "has_external": has_external,
        "external_tools": ext_tools,
    }


def determine_steps(obj, dep_graph, impact, proposed_name):
    vtypes = set(obj.violation_types)
    ds     = dep_graph["downstream_count"]
    us     = dep_graph["upstream_count"]
    owners = dep_graph["total_impacted_owners"][:5]
    bridge_days = 90 if impact["risk_level"] in ("critical", "high") else 30
    steps, n = [], 1

    if vtypes & _TAG_VIOLATIONS:
        steps.append({"step": n,
            "action": "Apply required tags via ALTER TABLE ... SET TAGS (owner, domain, quality_tier)",
            "automated": True, "owner": "data-platform-team"})
        n += 1

    if vtypes & _NAMING_VIOLATIONS:
        if ds > 0:
            owner_str = ", ".join(owners) if owners else "unknown"
            steps.append({"step": n,
                "action": f"Notify {ds} downstream consumer(s) of pending rename: {owner_str}",
                "automated": True, "owner": "data-platform-team"})
            n += 1
        if impact["has_pipeline"] or impact["has_job"]:
            steps.append({"step": n,
                "action": "Schedule maintenance window with pipeline/job owners before executing rename",
                "automated": False, "owner": "pipeline-owner"})
            n += 1
        steps.append({"step": n,
            "action": f"ALTER TABLE {obj.catalog_name}.{obj.schema_name}.{obj.table_name} RENAME TO {proposed_name or obj.table_name}",
            "automated": True, "owner": "data-platform-team"})
        n += 1
        if ds > 0:
            steps.append({"step": n,
                "action": f"CREATE OR REPLACE VIEW {obj.catalog_name}.{obj.schema_name}.{obj.table_name} AS SELECT * FROM {obj.catalog_name}.{obj.schema_name}.{proposed_name or obj.table_name}  -- bridge expires in {bridge_days}d",
                "automated": True, "owner": "data-platform-team"})
            n += 1

    if vtypes & _STALENESS_VIOLATIONS:
        if us == 0 and ds == 0:
            steps.append({"step": n,
                "action": "Zero lineage — evaluate for DROP after owner sign-off (no consumers at risk)",
                "automated": False, "owner": "table-owner"})
        else:
            steps.append({"step": n,
                "action": "Tag as deprecated; notify owner for retention decision",
                "automated": True, "owner": "data-platform-team"})
        n += 1

    if ds > 0:
        steps.append({"step": n,
            "action": f"Validate {ds} downstream consumer(s) post-change; remove bridge view after {bridge_days}d",
            "automated": False, "owner": "consumer-owners"})

    return steps

print("classify_impact and determine_steps ready.")


In [0]:
# AI rename suggestion — narrow scope, parallelized
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

_NAMING_VIOLATIONS = {"table_naming_violation", "table_anti_pattern", "schema_naming_violation"}

def _regex_rename(name):
    n = name.lower()
    n = re.sub(r"^[^a-z]+", "", n)
    n = re.sub(r"[-.\s]+", "_", n)
    n = re.sub(r"_(v?\d+|copy|backup|old|test|temp|bak)$", "", n)
    n = re.sub(r"[^a-z0-9_]", "", n)
    return n.strip("_") or "table_renamed"

# responseFormat with json_schema instructs the model to emit strict JSON —
# no text-stripping or split()[0] hacks needed.
_RENAME_RF = (
    '{"type":"json_schema","json_schema":{"name":"rename","strict":true,'
    '"schema":{"type":"object","properties":{"name":{"type":"string",'
    '"description":"Compliant snake_case name matching ^[a-z][a-z0-9_]{2,127}$"}},'
    '"required":["name"],"additionalProperties":false}}}'
)


def _suggest_one(table_name):
    """Single ai_query call for one rename suggestion. Falls back to regex."""
    prompt = (
        f"Table name '{table_name}' violates snake_case convention "
        f"(^[a-z][a-z0-9_]{{2,127}}$). Remove date/version/backup/temp suffixes. "
        f"Suggest one short compliant snake_case name."
    )
    try:
        # Sanitize prompt: collapse whitespace, escape single quotes
        safe = " ".join(prompt.replace("'", "''").replace("\\", "").splitlines())
        # model_name validated at startup via validate_model_name()
        # Use backtick-safe interpolation — model_name is alphanum+dot+hyphen only
        row  = spark.sql(
            f"SELECT ai_query('{model_name}', '{safe}', responseFormat => '{_RENAME_RF}') AS resp"
        ).first().resp
        name = json.loads(row)["name"].strip().lower()
        if re.match(r"^[a-z][a-z0-9_]{1,126}$", name):
            return name
    except Exception as e:
        print(f"  ⚠ ai_query rename failed for {table_name} ({type(e).__name__}): {e}")
    return _regex_rename(table_name)


def build_rename_map(objects, enable_ai):
    """
    Parallel rename suggestions for all objects needing a naming fix.
    Returns dict: table_name → proposed_name.
    Only calls AI for objects with naming violations; others get None.
    """
    needs_rename = [
        o for o in objects
        if set(o.violation_types) & _NAMING_VIOLATIONS
    ]
    if not needs_rename:
        return {}

    if not enable_ai:
        return {o.table_name: _regex_rename(o.table_name) for o in needs_rename}

    print(f"  Generating rename suggestions for {len(needs_rename)} tables in parallel...")
    result = {}
    with ThreadPoolExecutor(max_workers=min(len(needs_rename), 10)) as pool:
        futures = {pool.submit(_suggest_one, o.table_name): o.table_name for o in needs_rename}
        for fut in as_completed(futures):
            table_name = futures[fut]
            result[table_name] = fut.result()
    return result

print("Rename suggestion helpers ready.")


In [0]:
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, IntegerType, FloatType
)

_plan_schema = StructType([
    StructField("plan_id",            StringType()),
    StructField("created_at",         TimestampType()),
    StructField("catalog_name",       StringType()),
    StructField("schema_name",        StringType()),
    StructField("table_name",         StringType()),
    StructField("violation_type",     StringType()),   # pipe-sep list of all violation types
    StructField("proposed_new_name",  StringType()),
    StructField("upstream_count",     IntegerType()),
    StructField("downstream_count",   IntegerType()),
    StructField("upstream_entities",  StringType()),
    StructField("downstream_entities",StringType()),
    StructField("risk_level",         StringType()),
    StructField("estimated_effort",   StringType()),
    StructField("priority_score",     FloatType()),
    StructField("migration_steps",    StringType()),
    StructField("rollback_plan",      StringType()),
    StructField("plan_status",        StringType()),
    StructField("approved_by",        StringType()),
    StructField("approved_at",        TimestampType()),
    StructField("jira_ticket_id",     StringType()),
    StructField("jira_ticket_url",    StringType()),
    StructField("completed_at",       TimestampType()),
    StructField("completion_notes",   StringType()),
    StructField("violations_summary", StringType()),   # full JSON violations detail
])

# Fetch all lineage in parallel — UC REST API, not table scans
print("Loading lineage via UC API...")
lineage_map = build_lineage_map(objects_needing_plans)

# Parallel rename suggestions — all ai_query calls in-flight simultaneously
print("Generating rename suggestions...")
rename_map = build_rename_map(objects_needing_plans, enable_ai_plans)

plans_created = 0

for obj in objects_needing_plans:
    fqn = f"{obj.catalog_name}.{obj.schema_name}.{obj.table_name}"
    vtypes_str = " | ".join(sorted(obj.violation_types))
    print(f"\n{'='*60}")
    print(f"Table:  {fqn}")
    print(f"Issues: {vtypes_str}")

    # 1. Lineage (from pre-built UC API map)
    fqn_key = f"{obj.catalog_name}.{obj.schema_name}.{obj.table_name}"
    dep_graph = lineage_map.get(fqn_key, {
        "downstream": [], "upstream": [], "consumer_types": {},
        "downstream_count": 0, "upstream_count": 0, "total_impacted_owners": []
    })
    print(f"  ↑ upstream={dep_graph['upstream_count']}  ↓ downstream={dep_graph['downstream_count']}")
    print(f"  Consumer types: {dep_graph['consumer_types']}")

    # 2. Deterministic impact (includes external tool registry lookup)
    impact = classify_impact(dep_graph, fqn=fqn_key)
    if impact.get("has_external"):
        print(f"  ⚠ External tools registered: {', '.join(impact['external_tools'])}")
    print(f"  Risk: {impact['risk_level']}  Priority: {impact['priority_score']}")

    # 3. Proposed rename (from pre-built parallel map)
    proposed_name = rename_map.get(obj.table_name)
    if proposed_name:
        print(f"  Proposed rename: {proposed_name}")

    # 4. Deterministic steps
    steps = determine_steps(obj, dep_graph, impact, proposed_name or obj.table_name)
    rollback = (
        f"DROP VIEW {obj.catalog_name}.{obj.schema_name}.{obj.table_name}; "
        f"ALTER TABLE {obj.catalog_name}.{obj.schema_name}.{proposed_name} RENAME TO {obj.table_name}"
        if proposed_name else
        "Remove tags applied in step 1; no structural change was made"
    )

    violations_summary = json.dumps({
        "violation_types": sorted(obj.violation_types),
        "violation_details": list(obj.violation_details)[:20],
        "consumer_types": dep_graph["consumer_types"],
        "impacted_owners": dep_graph["total_impacted_owners"],
    })

    # 5. Write plan
    plan_id = str(uuid.uuid4())
    now = datetime.now(timezone.utc)

    try:
        plan_df = spark.createDataFrame([(
            plan_id, now,
            obj.catalog_name, obj.schema_name, obj.table_name,
            " | ".join(sorted(obj.violation_types)),
            proposed_name or "",
            dep_graph["upstream_count"],
            dep_graph["downstream_count"],
            json.dumps(dep_graph["upstream"][:20]),
            json.dumps(dep_graph["downstream"][:20]),
            impact["risk_level"],
            "deterministic",
            impact["priority_score"],
            json.dumps(steps),
            rollback,
            "proposed",
            None, None, None, None, None, None,  # approval / jira / completion fields
            violations_summary,
        )], _plan_schema)

        plan_df.createOrReplaceTempView("_tmp_plan")
        spark.sql(f"""
        INSERT INTO {safe_table_ref(catalog, control_schema, "migration_plans")}
        SELECT
          plan_id, created_at, catalog_name, schema_name, table_name,
          violation_type, proposed_new_name, upstream_count, downstream_count,
          upstream_entities, downstream_entities, risk_level, estimated_effort,
          priority_score, migration_steps, rollback_plan, plan_status,
          approved_by, approved_at, jira_ticket_id, jira_ticket_url,
          completed_at, completion_notes, violations_summary
        FROM _tmp_plan
        """)
        plans_created += 1
        print(f"  ✅ Plan written  steps={len(steps)}  risk={impact['risk_level']}")
    except Exception as e:
        print(f"  ⚠ Write failed: {e}")
        continue

print(f"\n\n✅ Created {plans_created} / {len(objects_needing_plans)} plans")


In [0]:
# Step 6: Execute approved plans — rename THEN bridge view at old name
# Correct lifecycle: rename table → create view at OLD fqn pointing to NEW fqn
# Bridge TTL is risk-aware: 90d for critical/high, 30d for medium/low
from datetime import datetime, timezone, timedelta
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import re as _re

_bridge_schema = StructType([
    StructField("bridge_view_fqn", StringType()),
    StructField("original_table_fqn", StringType()),
    StructField("new_table_fqn", StringType()),
    StructField("created_at", TimestampType()),
    StructField("expires_at", TimestampType()),
    StructField("plan_id", StringType()),
    StructField("status", StringType()),
])

approved_plans = spark.sql(f"""
SELECT * FROM {safe_table_ref(catalog, control_schema, "migration_plans")}
WHERE plan_status = 'approved'
""").collect()

bridges_created = 0
renames_executed = 0

for plan in approved_plans:
    fqn = f"{plan.catalog_name}.{plan.schema_name}.{plan.table_name}"
    new_fqn = f"{plan.catalog_name}.{plan.schema_name}.{plan.proposed_new_name}"

    steps = json.loads(plan.migration_steps) if plan.migration_steps else []

    # Validate identifiers once per plan (applies to all steps)
    try:
        validate_identifier(plan.catalog_name, "plan.catalog_name")
        validate_identifier(plan.schema_name, "plan.schema_name")
        validate_identifier(plan.table_name, "plan.table_name")
        if plan.proposed_new_name:
            validate_identifier(plan.proposed_new_name, "plan.proposed_new_name")
    except ValueError as e:
        print(f"  ⚠ Skipping plan {plan.plan_id}: {e}")
        continue

    # Parse risk-aware bridge TTL from plan's risk_level
    bridge_days = 90 if plan.risk_level in ("critical", "high") else 30

    # Two-pass execution: renames first, then bridges
    # This ensures the bridge target exists before we create the view.
    rename_succeeded = False

    # Pass 1: Execute rename steps
    for step in steps:
        if not step.get("automated", False):
            continue
        action = step.get("action", "")
        action_lower = action.lower()

        if "alter table" in action_lower and "rename to" in action_lower:
            try:
                _old_ref = safe_table_ref(plan.catalog_name, plan.schema_name, plan.table_name)
                _new_ref = safe_table_ref(plan.catalog_name, plan.schema_name, plan.proposed_new_name)
                spark.sql(f"ALTER TABLE {_old_ref} RENAME TO {_new_ref}")
                renames_executed += 1
                rename_succeeded = True
                print(f"  ✅ Renamed: {fqn} → {new_fqn}")
            except Exception as e:
                print(f"  ⚠ Rename failed for {fqn}: {e}")
                # If rename fails, skip bridge creation — it would be self-referential
                break

    # Pass 2: Create bridge views (only if rename succeeded)
    for step in steps:
        if not step.get("automated", False):
            continue
        action = step.get("action", "")
        action_lower = action.lower()

        if ("bridge" in action_lower or "create or replace view" in action_lower) and "rename to" not in action_lower:
            if not rename_succeeded:
                print(f"  ⚠ Skipping bridge for {fqn}: rename did not succeed")
                continue
            try:
                # Bridge view sits at the OLD name, pointing to the NEW location
                _bridge_ref = safe_table_ref(plan.catalog_name, plan.schema_name, plan.table_name)
                _target_ref = safe_table_ref(plan.catalog_name, plan.schema_name, plan.proposed_new_name)
                spark.sql(f"CREATE OR REPLACE VIEW {_bridge_ref} AS SELECT * FROM {_target_ref}")

                now = datetime.now(timezone.utc)
                bridge_df = spark.createDataFrame(
                    [(
                        fqn,          # bridge_view_fqn = old name (where the view lives)
                        fqn,          # original_table_fqn = what consumers knew
                        new_fqn,      # new_table_fqn = where data actually lives now
                        now,
                        now + timedelta(days=bridge_days),  # risk-aware TTL
                        plan.plan_id,
                        "active",
                    )],
                    _bridge_schema,
                )
                bridge_df.createOrReplaceTempView("_tmp_bridge")
                spark.sql(f"""
                INSERT INTO {safe_table_ref(catalog, control_schema, "bridge_views")}
                SELECT bridge_view_fqn, original_table_fqn, new_table_fqn,
                       created_at, expires_at, plan_id, status
                FROM _tmp_bridge
                """)
                bridges_created += 1
                print(f"  🌉 Bridge: {fqn} → {new_fqn} (expires in {bridge_days}d)")
            except Exception as e:
                print(f"  ⚠ Bridge creation failed for {fqn}: {e}")

    # Update plan status to in_progress
    validate_uuid(plan.plan_id, "plan.plan_id")
    spark.sql(f"""
    UPDATE {safe_table_ref(catalog, control_schema, "migration_plans")}
    SET plan_status = 'in_progress'
    WHERE plan_id = '{plan.plan_id}'
    """)

print(f"\nRenames executed: {renames_executed}")
print(f"Bridge views created: {bridges_created}")
print(f"Plans moved to in_progress: {len(approved_plans)}")

In [0]:
# ── Summary & observability ──────────────────────────────────────────────────
try:
    plans_written = len(batch_rows)
except NameError:
    plans_written = 0

# Feature check summary
_fc_passed  = sum(1 for r in feature_results if r.status.value == 'PASSED') if 'feature_results' in dir() else 0
_fc_failed  = sum(1 for r in feature_results if r.status.value == 'FAILED') if 'feature_results' in dir() else 0
_fc_skipped = sum(1 for r in feature_results if r.status.value == 'SKIPPED') if 'feature_results' in dir() else 0
_fc_total   = _fc_passed + _fc_failed + _fc_skipped
_fc_score   = f"{_fc_passed / (_fc_passed + _fc_failed) * 100:.0f}%" if (_fc_passed + _fc_failed) > 0 else "N/A"

print(f"""
{'='*52}
  RECONCILIATION PLANNER COMPLETE
{'='*52}
  Violations queued:   {len(objects_needing_plans_list) if 'objects_needing_plans_list' in dir() else 'N/A'}
  Plans written:       {plans_written}
  Bridge views:        {bridges_created if 'bridges_created' in dir() else 0}
  AI enabled:          {enable_ai_plans}
  Jira:                {'enabled' if jira_base_url and jira_api_token else 'disabled'}
  Feature checks:      {_fc_total} ({_fc_passed} passed, {_fc_failed} failed, {_fc_skipped} skipped)
  Feature score:       {_fc_score}
  Control schema: {catalog}.{control_schema}
{'='*52}
""")

try:
    spark.sql(f"""
    INSERT INTO {safe_table_ref(catalog, control_schema, "job_run_history")} VALUES (
      CURRENT_DATE(),
      'uc_hygiene_daily_governance',
      'p3_reconciliation',
      'p3_remediation',
      'success',
      {plans_written},
      {plans_written},
      {plans_written},
      int(_t.time() - _task_start),
      'plans_written={plans_written} ai={'enabled' if enable_ai_plans else 'disabled'} jira={'enabled' if jira_base_url and jira_api_token else 'disabled'}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")